## Setup and imports

In [ ]:
EXP_NAME = "SPS_audit_100_with_ind"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'

In [ ]:
print(PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1.2)

sps_audit = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit.csv')
sbs_audit_baseline = pd.read_csv(f'{PROJECT_ROOT}/results/{EXP_NAME}/sps_audit_baseline.csv')
dataset = pd.read_csv(f'{PROJECT_ROOT}/data/heart_disease_cleaned.csv')

# Analysis

### Basics

In [ ]:
scorings = sps_audit.groupby('feature')[['sensitivity_scoring', 'fidelity_scoring']].first()

sps_audit = sps_audit.drop(['sensitivity_scoring', 'fidelity_scoring'], axis=1)

In [ ]:
iteration_per_feature = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('feature')[['roc_auc']].count()
iteration_per_feature

### Counterfactual sensitivity

In [ ]:
sps_audit.groupby('feature')['cf_sensitivity'].agg(['mean','std','median']).sort_values(by='mean')

In [ ]:
# Counterfactual sensitivity
f, ax = plt.subplots(figsize=(8, 6))
sns.boxplot(data=sps_audit, x="feature", y="cf_sensitivity", showmeans=True, 
            meanprops={"marker": "+",
                       "markeredgecolor": "black",
                       "markersize": "8"}, ax=ax)

plt.plot()


# Utility vs Fairness trade-off

In [ ]:
# sps_audit.head()

In [ ]:
baseline_roc_auc = sbs_audit_baseline['roc_auc'].values[0]
baseline_ieco_mace = sbs_audit_baseline['ieco_mace'].values[0]

print(f"Baseline ROC AUC: {baseline_roc_auc}")
print(f"Baseline IECO MACE: {baseline_ieco_mace}")

## Pareto Frontier 


In [ ]:
feature_desc_stats = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby(['feature'])[['ieco_mace', 'auprc', 'roc_auc']].median()
feature_corr_stats = sps_audit[sps_audit['bucket'] == 'x_corr'].groupby(['feature'])[['ieco_mace', 'auprc', 'roc_auc']].median()

In [ ]:
utility_vs_fairness = sps_audit.groupby('iteration')[['roc_auc', 'auprc','ieco_mace']].first().sort_values('ieco_mace')
x_desc_configs = sps_audit[sps_audit['bucket'] == 'x_desc'].groupby('iteration')['feature'].apply(list).to_dict()

positives = dataset[dataset['cvd'] == 1]
auprc_baseline = len(positives) / len(dataset)

pareto_frontier = []
current_max_utility = -1

print('--- Configurations on the AUPRC Pareto Frontier ---')
for idx, solution in utility_vs_fairness.iterrows():
  if solution['auprc'] > current_max_utility:
    pareto_frontier.append(solution)
    current_max_utility = solution['auprc']
    print(f'Iteration {idx}, Xdesc: {x_desc_configs[idx]}')
pareto_frontier_df = pd.DataFrame(pareto_frontier)
print(pareto_frontier_df.to_markdown())


fig, axes = plt.subplots(1, 2,  figsize=(20, 6))
axes[0].plot(baseline_roc_auc, baseline_ieco_mace, marker="D", color="#D92B89", linestyle="")
axes[1].plot(baseline_roc_auc, baseline_ieco_mace, marker="D", color="#D92B89", linestyle="")
sns.lineplot(data=pareto_frontier_df, x='auprc', y='ieco_mace', color="green", marker='', linestyle="--", errorbar=None, ax=axes[0])
sns.lineplot(data=pareto_frontier_df, x='auprc', y='ieco_mace', color="green", linestyle="--", errorbar=None, ax=axes[1])
sns.scatterplot(data=utility_vs_fairness, x='auprc', y='ieco_mace', ax=axes[0])
sns.scatterplot(data=feature_desc_stats, x='auprc', y='ieco_mace', hue='feature', color='', ax=axes[1])
# plt.axvline(auprc_baseline)
plt.xlabel('Average Precision (AUPRC)')
plt.ylabel('IECO MACE')
plt.legend(labels=[ 
                   'Bias-unaware configuration',
                   'Pareto frontier', 
                   'CEVAE-HE result for each pathway configuration in the audit'
                   ]+feature_desc_stats.index.to_list(),
           loc='upper left', bbox_to_anchor=(1, 1), edgecolor="white")

plt.show()

In [ ]:
feature_desc_stats['ieco_baseline_delta'] = feature_desc_stats['ieco_mace'] - baseline_ieco_mace
feature_desc_stats['auprc_baseline_delta'] = feature_desc_stats['auprc'] - baseline_roc_auc
feature_desc_stats['roc_auc_baseline_delta'] = feature_desc_stats['roc_auc'] - baseline_roc_auc 
feature_desc_stats['ieco_corr_delta'] = feature_desc_stats['ieco_mace'] - feature_corr_stats['ieco_mace']
feature_desc_stats['auprc_corr_delta'] = feature_desc_stats['auprc'] - feature_corr_stats['auprc']
feature_desc_stats['roc_auc_corr_delta'] = feature_desc_stats['roc_auc'] - feature_corr_stats['roc_auc']

feature_desc_stats = feature_desc_stats.sort_values('ieco_baseline_delta')
print(feature_desc_stats.to_markdown())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))
sns.scatterplot(data=feature_desc_stats, x='ieco_baseline_delta', y='auprc_baseline_delta', hue='feature', ax=axes[0,0])
sns.scatterplot(data=feature_desc_stats, x='ieco_baseline_delta', y='roc_auc_baseline_delta', hue='feature', ax=axes[1,0])
sns.scatterplot(data=feature_desc_stats, x='ieco_corr_delta', y='auprc_corr_delta', hue='feature', ax=axes[0,1])
sns.scatterplot(data=feature_desc_stats, x='ieco_corr_delta', y='roc_auc_corr_delta', hue='feature', ax=axes[1,1])
plt.plot()